In [ ]:
import os
import sys
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipyaggrid import Grid
from IPython.display import display, HTML, clear_output

# Setup path for imports
notebook_dir = Path('.').resolve()
sys.path.insert(0, str(notebook_dir.parent))

from utilities import (
    SPINNER_HTML, COMMON_CSS, DASHBOARD_WIDTH,
    PRODUCT_INFO_COLUMNS, VOLUME_AND_WEIGHTS_COLUMNS,
    BS_COMPONENTS_PACKAGE, 
    NOMADAPIClient, normalize_sample_id,
    make_row_by_key, wait_for_sample_ids,
    resolve_name_to_reference, resolve_reference_to_name,
    extract_grid_frame, validate_row, render_status,
)

display(HTML(COMMON_CSS))

In [ ]:
# API Configuration
URL_BASE = 'https://nomad08.csn29.bessy.de'
URL = f'{URL_BASE}/nomad-oasis/api/v1'
TOKEN = os.environ.get('NOMAD_CLIENT_ACCESS_TOKEN', '')
HEADERS = {
    'Authorization': f'Bearer {TOKEN}',
    'Accept': 'application/json',
}

# Initialize API client
api_client = NOMADAPIClient(URL_BASE, HEADERS)

PROCESS_SETTLE_SECONDS = 2
PROCESS_RETRY_COUNT = 2

In [ ]:
# Base schema definition for ElectrolyteSample
BASE_SCHEMA = {
    'entry_type': 'Electrolyte Sample',
    'm_def': f'{BS_COMPONENTS_PACKAGE}.ElectrolyteSample',
    'required_keys': ['name'],
}

# Base columns (always visible)
BASE_COLUMNS = [
    ('name', 'Sample Name *', 'str'),
    ('notes', 'Notes', 'str'),
    ('tags', 'Tags', 'str'),
    ('electrolyte_stock', 'Electrolyte Stock (Reference)', 'str'),
]

# Volume and weights columns (always visible for electrolyte samples)
BASE_COLUMNS.extend(VOLUME_AND_WEIGHTS_COLUMNS)


def get_columns_for_shape_and_product_info(include_product_info):
    """Build column list based on product info preference."""
    columns = BASE_COLUMNS.copy()
    
    if include_product_info:
        columns.extend(PRODUCT_INFO_COLUMNS)
    
    return columns

# Initial schema (will be updated dynamically)
SCHEMAS = {
    'Electrolyte Sample': {**BASE_SCHEMA, 'columns': BASE_COLUMNS},
}

In [ ]:
# ===== WIDGET INITIALIZATION =====
# Load uploads
uploads = api_client.get_uploads()
grid = None
col_keys = []
col_labels = []

upload_dd = widgets.Dropdown(
    options=[u.get('upload_name') for u in uploads if u.get('upload_name')],
    value=None,
    layout=widgets.Layout(width='300px', height='40px'),
    style={'description_width': 'initial'},
)
schema_dd = widgets.Dropdown(
    options=list(SCHEMAS.keys()),
    value='Electrolyte Sample',
    layout=widgets.Layout(width='300px', height='40px'),
    style={'description_width': 'initial'},
    disabled=True,
)
date_picker = widgets.NaiveDatetimePicker(
    value=datetime.now(), 
    disabled=False,
    style={'description_width': 'initial', 'font_size': '16px'}
)

# Product Info selection (shape not needed for electrolyte)
product_info_cb = widgets.Checkbox(
    value=False,
    description='Include Product Information?',
    indent=False,
)

out_grid = widgets.Output()
out_existing = widgets.Output()
out_status = widgets.Output()

In [ ]:
def prepare_rows_from_frame(frame, schema_name, schema, existing_ids):
    """Prepare rows from grid data for upload"""
    prepared_rows = []
    row_errors = []
    
    include_product_info = product_info_cb.value

    for row_number, (_row_index, row) in enumerate(frame.iterrows(), start=1):
        row_by_key = make_row_by_key(row, col_keys, col_labels)
        if not any(value is not None for value in row_by_key.values()):
            continue

        validation_error = validate_row(schema_name, schema, row_by_key)
        if validation_error:
            if row_by_key.get('name'):
                sample_name = row_by_key.get('name')
                row_errors.append(f'❌ {sample_name}: {validation_error}')
            else:
                row_errors.append(f'❌ row {row_number}: {validation_error}')
            continue

        sample_name = normalize_sample_id(row_by_key.get('name'))

        # Build volume_and_weights subsection
        volume_and_weights = {}
        if row_by_key.get('volume_ml'):
            volume_and_weights['volume'] = float(row_by_key['volume_ml'])
        if row_by_key.get('mass_g'):
            volume_and_weights['mass'] = float(row_by_key['mass_g'])

        # Build product_info subsection (if enabled)
        product_info = {}
        if include_product_info:
            if row_by_key.get('product_number'):
                product_info['product_number'] = row_by_key['product_number']
            if row_by_key.get('lot_number'):
                product_info['lot_number'] = row_by_key['lot_number']
            if row_by_key.get('product_volume'):
                product_info['product_volume'] = float(row_by_key['product_volume'])
            if row_by_key.get('product_weight'):
                product_info['product_weight'] = float(row_by_key['product_weight'])
            if row_by_key.get('shipping_date'):
                product_info['shipping_date'] = row_by_key['shipping_date']
            if row_by_key.get('opening_date'):
                product_info['opening_date'] = row_by_key['opening_date']
            if row_by_key.get('supplier'):
                product_info['supplier'] = row_by_key['supplier']
            if row_by_key.get('product_description'):
                product_info['product_description'] = row_by_key['product_description']
            if row_by_key.get('cost'):
                product_info['cost'] = float(row_by_key['cost'])

        data = {
            'm_def': schema['m_def'],
            'name': sample_name,
            'datetime': date_picker.value.strftime('%Y-%m-%dT%H:%M:%S.%f'),
            # Initialize substance_identifiers subsection to allow normalize() to populate it
            'substance_identifiers': {},
        }

        if volume_and_weights:
            data['volume_and_weights'] = volume_and_weights
        if product_info:
            data['product_info'] = product_info
        if row_by_key.get('notes'):
            data['notes'] = {'description': row_by_key['notes']}

        # Handle electrolyte_stock reference
        if row_by_key.get('electrolyte_stock'):
            elec_stock_input = row_by_key['electrolyte_stock']
            elec_stock_m_def = f'{BS_COMPONENTS_PACKAGE}.ElectrolyteStock'
            # If it's already a reference (contains /archive/), use as-is
            # Otherwise, treat as a name and convert to reference
            if '/archive/' in str(elec_stock_input):
                data['electrolyte_stock'] = elec_stock_input
            else:
                # Try to resolve the name to a reference 
                elec_stock_ref = resolve_name_to_reference(api_client, elec_stock_input, elec_stock_m_def)
                if elec_stock_ref:
                    data['electrolyte_stock'] = elec_stock_ref
                else:
                    row_errors.append(f"{sample_name}: Could not find ElectrolyteStock named '{elec_stock_input}'")

        # Parse tags from comma-separated string to array
        tags_input = row_by_key.get('tags')
        if tags_input:
            tags_list = [tag.strip() for tag in str(tags_input).split(',') if tag.strip()]
            if tags_list:
                data['tags'] = tags_list


        file_name = f"{str(sample_name).replace(' ', '_')}.archive.json"
        prepared_rows.append({
            'row_number': row_number,
            'name': sample_name,
            'is_new': sample_name not in existing_ids,
            'file_name': file_name,
            'raw_path': file_name,
            'row_by_key': row_by_key,
            'archive': {'data': data},
        })

    return prepared_rows, row_errors

In [ ]:
def render_existing_samples(upload_id, schema):
    """Return HTML widget for displaying existing samples."""
    existing = api_client.get_existing_samples(upload_id, schema) if upload_id else []
    
    if not existing:
        return HTML('<p style="color: #999; font-style: italic;">No existing samples in this upload</p>')
    
    # Build existing entries table
    existing_data = []
    for sample in existing:
        data = sample.get('archive', {}).get('data', {})
        vol_weights = data.get('volume_and_weights', {})
        product_info = data.get('product_info', {}) if isinstance(data.get('product_info'), dict) else {}
        notes = data.get('notes', {}) if isinstance(data.get('notes'), dict) else {}
        
        # Resolve electrolyte_stock reference to name
        elec_stock_ref = data.get('electrolyte_stock')
        elec_stock_display = None
        if elec_stock_ref:
            elec_stock_name = resolve_reference_to_name(api_client, elec_stock_ref)[1]
            elec_stock_display = elec_stock_name if elec_stock_name else elec_stock_ref
        
        # Format tags array as comma-separated string
        tags_array = data.get('tags', [])
        tags_display = ', '.join(tags_array) if tags_array else ''
        
        row = {
            'Lab ID': data.get('lab_id', ''),
            'Sample Name': data.get('name', 'N/A'),
            'Notes': notes.get('description') if notes.get('description') else '',
            'Tags': tags_display,
            'Electrolyte Stock': elec_stock_display,
            'Volume [ml]': vol_weights.get('volume') if isinstance(vol_weights, dict) else None,
            'Mass [g]': vol_weights.get('mass') if isinstance(vol_weights, dict) else None,
            'Product Number': product_info.get('product_number'),
            'Lot Number': product_info.get('lot_number'),
            'Product Volume [ml]': product_info.get('product_volume'),
            'Product Weight [g]': product_info.get('product_weight'),
            'Shipping Date [YYYY-MM-DD]': product_info.get('shipping_date'),
            'Opening Date [YYYY-MM-DD]': product_info.get('opening_date'),
            'Supplier': product_info.get('supplier'),
            'Product Description': product_info.get('product_description'),
            'Cost [EUR]': product_info.get('cost'),
        }
        existing_data.append(row)
    
    df_existing = pd.DataFrame(existing_data)
    
    # Build HTML table manually with left-aligned text
    header_html = ''.join([
        f'<th style="text-align: left; padding: 8px; border-bottom: 2px solid #ccc; font-weight: 600; white-space: nowrap;">{col}</th>'
        for col in df_existing.columns
    ])
    
    rows_html = ''
    for _, row in df_existing.iterrows():
        cells = ''.join([
            f'<td style="text-align: left; padding: 6px 8px; border-bottom: 1px solid #eee;">{str(val) if val is not None else ""}</td>'
            for val in row
        ])
        rows_html += f'<tr>{cells}</tr>'
    
    html_table = f'''
    <div style="background: #f0f5ff; padding: 15px; border-radius: 8px; margin-bottom: 20px; border: 1px solid #ddd;">
        <h4 style="margin-top: 0; color: #333;">📋 Existing Samples in Upload</h4>
        <div style="overflow-x: auto; max-height: 300px; overflow-y: auto;">
            <table style="border-collapse: collapse; width: 100%; font-size: 0.9em; table-layout: auto;">
                <thead><tr>{header_html}</tr></thead>
                <tbody>{rows_html}</tbody>
            </table>
        </div>
    </div>
    '''
    return HTML(html_table)

In [ ]:
def show_grid():
    """Display existing samples (read-only) and new entry grid (editable)."""
    global grid, col_keys, col_labels
    
    schema_name = schema_dd.value
    schema = SCHEMAS[schema_name]
    upload_id = api_client.get_upload_id(uploads, upload_dd.value)
    
    # Get selected options
    include_product_info = product_info_cb.value
    
    # Update schema columns dynamically
    schema['columns'] = get_columns_for_shape_and_product_info(include_product_info)
    
    col_keys = [column[0] for column in schema['columns']]
    col_labels = [column[1] for column in schema['columns']]
    
    # Display existing samples (read-only)
    out_existing.clear_output()
    with out_existing:
        widget_html = render_existing_samples(upload_id, schema)
        display(widget_html)
    
    # Get available ElectrolyteStock names for dropdown
    elec_stock_m_def = f'{BS_COMPONENTS_PACKAGE}.ElectrolyteStock'
    try:
        all_entries = api_client.iter_archive_query({})
        electrolyte_stock_names = sorted([
            entry.get('archive', {}).get('data', {}).get('name')
            for entry in all_entries
            if (entry.get('archive', {}).get('data', {}).get('m_def') == elec_stock_m_def
                and entry.get('archive', {}).get('data', {}).get('name'))
        ])
        if not electrolyte_stock_names:
            print(f"⚠️ No ElectrolyteStock entries found (queried {len(all_entries)} total entries)")
    except Exception as e:
        print(f"⚠️ Could not load ElectrolyteStock entries: {e}")
        electrolyte_stock_names = []
    
    # Create new entry grid
    df = pd.DataFrame(columns=col_labels)
    
    blank_rows = 12
    for _index in range(blank_rows):
        df.loc[len(df)] = pd.Series(dtype='object')
    
    # Build column definitions with select editor for electrolyte_stock
    column_defs = []
    for label in df.columns:
        col_def = {'headerName': label, 'field': label}
        
        # Add dropdown for Electrolyte Stock column
        if label == 'Electrolyte Stock (Reference)':
            col_def['cellEditor'] = 'agSelectCellEditor'
            # Add empty string at start to allow clearing the selection
            values_with_clear = [''] + electrolyte_stock_names if electrolyte_stock_names else ['']
            col_def['cellEditorParams'] = {
                'values': values_with_clear
            }
        
        elif label.startswith('Notes'):
            col_def['cellEditor'] = 'agLargeTextCellEditor'
                
        elif label.startswith('Tags'):
            col_def['cellEditor'] = 'agLargeTextCellEditor'
                
        # Add date picker for Shipping Date and Opening Date columns
        elif label.startswith('Shipping Date') or label.startswith('Opening Date'):
            col_def['cellEditor'] = 'agDateCellEditor'
            col_def['cellEditorParams'] = {
                'format': 'YYYY-MM-DD'
            }
            col_def['filter'] = 'agDateColumnFilter'
        
        column_defs.append(col_def)
    
    grid_options = {
        'columnDefs': column_defs,
        'defaultColDef': {'editable': True, 'resizable': True},
        'rowSelection': 'multiple',
        'enableRangeSelection': True,
        'stopEditingWhenCellsLoseFocus': True,
    }
    grid = Grid(
        grid_data=df,
        grid_options=grid_options,
        sync_on_edit=True,
        theme='ag-theme-balham',
        columns_fit='auto',
        index=False,
    )
    
    out_grid.clear_output()
    with out_grid:
        display(grid)

In [ ]:
def on_create(_button):
    """Handle click on Create button.
    
    NOTE: This implementation is identical in all notebooks.    
        If you modify this function, update other notebooks to keep them in sync.
        The only difference between notebooks is the prepare_rows_from_frame() implementation.
    """
    with out_status:
        clear_output(wait=True)
        
        schema_name = schema_dd.value
        schema = SCHEMAS[schema_name]
        upload_id = api_client.get_upload_id(uploads, upload_dd.value)
        
        # Validation: upload selected
        if not upload_id:
            display(render_status('error', '❌ No upload selected'))
            return
        
        # Validation: grid data
        frame = extract_grid_frame(grid, col_labels)
        if frame.empty:
            display(render_status('error', '❌ No data in grid'))
            return
        
        # Get existing samples
        existing = api_client.get_existing_samples(upload_id, schema)
        existing_ids = {
            normalize_sample_id(s.get('archive', {}).get('data', {}).get('name'))
            for s in existing
            if s.get('archive', {}).get('data', {})
        }
                
        # Prepare rows
        prepared_rows, row_errors = prepare_rows_from_frame(frame, schema_name, schema, existing_ids)
        
        if not prepared_rows:
            if row_errors:
                display(render_status('error', f'{"<br/>".join(row_errors)}'))
            else:
                display(render_status('warning', '⚠️  No data rows to submit'))
            return
        
        if row_errors:
            display(render_status('warning', f'⚠️  Some rows had errors and were skipped:<br/>{"<br/>".join(row_errors)}'))
        
        # Track messages to re-display after clearing spinners
        messages = []
        
        # Upload
        display(HTML(SPINNER_HTML.format(msg='Uploading archive data...')))
        status, detail = api_client.write_archive_bundle_via_api(upload_id, prepared_rows)
        clear_output(wait=True)
        
        if status != 200:
            display(render_status('error', f'❌ Upload failed: {status}<br/>{detail}'))
            return
        
        upload_msg = render_status('success', f'✅ Uploaded {len(prepared_rows)} samples')
        display(upload_msg)
        messages.append(upload_msg)
        
        # Process upload
        display(HTML(SPINNER_HTML.format(msg='Processing upload...')))
        last_payload = None
        process_state_info = ''
        try:
            last_payload = api_client.process_upload(upload_id, timeout=180)
            
            # Get the processing state
            state_value = (
                last_payload.get('current_process')
                or last_payload.get('process_status')
                or last_payload.get('last_status_message')
                or 'completed'
            )
            process_state_info = f'<br/>State: {state_value}'
            process_msg = render_status('success', f'✅ Upload processed successfully!{process_state_info}')
            display_error = False
        except TimeoutError as e:
            process_msg = render_status('warning', f'⚠️  {e}')
            display_error = True
        except Exception as e:
            process_msg = render_status('error', f'❌ Processing failed: {e}')
            display_error = True
        
        clear_output(wait=True)
        
        # Re-display upload message and new process message
        for msg in messages:
            display(msg)
        display(process_msg)
        
        if display_error and 'Processing failed' in str(process_msg.data):
            return
        
        messages.append(process_msg)
        
        # Wait for samples to appear
        display(HTML(SPINNER_HTML.format(msg='Waiting for samples to register...')))
        time.sleep(PROCESS_SETTLE_SECONDS)
        new_sample_ids = [row['name'] for row in prepared_rows]
        
        try:
            wait_for_sample_ids(api_client, upload_id, schema, new_sample_ids, timeout=60)
            wait_msg = render_status('success', '✅ All samples registered and processed!')
        except TimeoutError:
            wait_msg = render_status('warning', '⚠️  Samples may still be processing, please refresh')
        
        clear_output(wait=True)
        
        # Re-display all messages
        for msg in messages:
            display(msg)
        display(wait_msg)
        
        # Refresh grid
        time.sleep(1)
        show_grid()


In [ ]:
def on_context_change(_change):
    out_status.clear_output()
    show_grid()

upload_dd.observe(on_context_change, names='value')
schema_dd.observe(on_context_change, names='value')
product_info_cb.observe(on_context_change, names='value')

btn_create = widgets.Button(
    description='✅ Upload & Process',
    button_style='success',
    layout=widgets.Layout(width='180px', height='40px'),
    tooltip='Create sample entries from the data grid and trigger NOMAD processing',
)
btn_refresh = widgets.Button(
    description='🔄 Refresh',
    layout=widgets.Layout(width='110px', height='40px'),
    tooltip='Reload existing entries from NOMAD',
)

btn_create.on_click(on_create)
btn_refresh.on_click(lambda _: show_grid())

# Build the dashboard layout
init_message = widgets.HTML(
    '<h3 style="background: #e3f2fd; padding: 12px 16px; border-radius: 8px; border-left: 4px solid #1976d2; color: #0d47a1;">'
    'ℹ️ Please select an upload to begin'
    '</h3>'
)

controls = widgets.VBox([
    widgets.HTML('<h3>Configuration</h3>'),
    widgets.HBox([
        widgets.HTML('<div style="width: 180px; padding-top: 8px; font-weight: 500;">Sample Type:</div>'),
        schema_dd
    ]),
    widgets.HBox([
        widgets.HTML('<div style="width: 180px; padding-top: 8px; font-weight: 500;">NOMAD Upload:</div>'),
        upload_dd
    ]),
    widgets.HBox([
        widgets.HTML('<div style="width: 180px; padding-top: 8px; font-weight: 500;">Sample Timestamp:</div>'),
        date_picker
    ]),
])

new_samples_message = widgets.HTML(
    '<h3 style="background: #e3f2fd; color: #0d47a1; padding: 12px 16px; border-radius: 8px; border-left: 4px solid #1976d2; margin: 15px 0 8px; font-size: 1.3em;">📝 Register new Samples &nbsp;<small style="color:#0d47a1">- edit cells directly, add rows at bottom</small></h3>'
)

data_options = widgets.VBox([
    widgets.HTML('<h4 style="margin-bottom: 8px;">Data Options</h4>'),
    product_info_cb,
])

buttons = widgets.HBox([btn_create, widgets.HTML('&nbsp;&nbsp;'), btn_refresh])

dashboard_banner = widgets.HTML(
    f'<div style="background: linear-gradient(135deg, #1a1a2e, #16213e); padding: 20px; border-radius: 10px; color: white; margin-bottom: 10px; width: {DASHBOARD_WIDTH}; box-sizing: border-box;">'
    '<div style="margin:0; font-family: monospace; font-size: 2.2em;">🔋 Electrolyte Sample Batch Registration</div>'
    '<p style="margin:4px 0 0; opacity:0.6; font-size:1rem;">Voila Dashboard for a batch registration of Electrolyte Sample entries in NOMAD</p>'
    '</div>'
)


dashboard_content = widgets.VBox([
    init_message,
    controls,
    out_existing,
    new_samples_message,
    data_options,
    out_grid,
    widgets.HTML('<div style="margin:8px 0; padding:10px; background:#fff3cd; border-radius:4px; border-left:4px solid #ffc107; color:#856404;">'
                 '<strong>⚠️ After editing:</strong> Click on an empty cell in the grid before pressing "Upload & Process"'
                 '</div>'),
    buttons,
    out_status,
])

# Display the complete dashboard with max-width container
display(dashboard_banner)
display(widgets.VBox([
            dashboard_content,
        ], 
        layout=widgets.Layout(width=DASHBOARD_WIDTH, margin='0')))